# ⏰ Listening Patterns

Wann hörst du Musik? Analyse nach Tageszeit, Wochentag und Platform.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from config import *
from data_loader import load_data, get_music, ensure_results_dir

plt.style.use(PLOT_STYLE)
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI
plt.rcParams['savefig.bbox'] = 'tight'

In [ ]:
df = load_data()
music = get_music(df)

## Heatmap: Wochentag × Tageszeit

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_labels = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']

heatmap_data = (music.groupby(['weekday', 'hour'])['minutes_played'].sum()
                .reset_index()
                .pivot(index='weekday', columns='hour', values='minutes_played')
                .reindex(weekday_order)
                .fillna(0))

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='Greens', ax=ax, fmt='.0f',
            yticklabels=weekday_labels, linewidths=0.5,
            cbar_kws={'label': 'Minuten (gesamt)'})
ax.set_xlabel('Uhrzeit')
ax.set_ylabel('Wochentag')
ax.set_title('🗓️ Hörzeit nach Wochentag und Uhrzeit (alle Jahre)', fontsize=16, fontweight='bold')

out = ensure_results_dir('overview')
fig.savefig(out / 'heatmap_weekday_hour.png')
plt.show()

## Heatmap pro Jahr

In [ ]:
years = sorted(music['year'].unique())

for year in years:
    year_music = music[music['year'] == year]
    hm = (year_music.groupby(['weekday', 'hour'])['minutes_played'].sum()
          .reset_index()
          .pivot(index='weekday', columns='hour', values='minutes_played')
          .reindex(weekday_order)
          .fillna(0))

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(hm, cmap='Greens', ax=ax, yticklabels=weekday_labels, linewidths=0.5,
                cbar_kws={'label': 'Minuten'})
    ax.set_xlabel('Uhrzeit')
    ax.set_ylabel('')
    ax.set_title(f'🗓️ Hörzeit-Heatmap {year}', fontsize=14, fontweight='bold')

    year_out = ensure_results_dir(year)
    fig.savefig(year_out / 'heatmap_weekday_hour.png')
    plt.show()

## Hörzeit nach Tageszeit (Durchschnitt)

In [ ]:
hourly = music.groupby('hour')['minutes_played'].mean().reset_index()

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
ax.fill_between(hourly['hour'], hourly['minutes_played'], alpha=0.3, color=COLOR_PRIMARY)
ax.plot(hourly['hour'], hourly['minutes_played'], color=COLOR_PRIMARY, linewidth=2.5, marker='o')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45)
ax.set_xlabel('Uhrzeit')
ax.set_ylabel('Ø Minuten')
ax.set_title('⏰ Durchschnittliche Hörzeit nach Tageszeit', fontsize=16, fontweight='bold')
fig.savefig(out / 'avg_minutes_by_hour.png')
plt.show()

## Platform-Verteilung

In [ ]:
platform = music.groupby('platform')['hours_played'].sum().sort_values(ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(platform)), platform['hours_played'], color=COLOR_PALETTE[:len(platform)])
ax.set_yticks(range(len(platform)))
ax.set_yticklabels(platform['platform'])
ax.invert_yaxis()
for bar, h in zip(bars, platform['hours_played']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{h:,.1f}h', va='center', fontsize=10)
ax.set_xlabel('Stunden')
ax.set_title('📱 Hörzeit nach Platform', fontsize=16, fontweight='bold')
plt.tight_layout()
fig.savefig(out / 'platform_distribution.png')
plt.show()

## Skip-Rate pro Jahr

In [ ]:
skip_rate = (music.groupby('year')
             .agg(total=('skipped', 'count'), skipped=('skipped', 'sum'))
             .reset_index())
skip_rate['skip_pct'] = (skip_rate['skipped'] / skip_rate['total']) * 100

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
bars = ax.bar(skip_rate['year'].astype(str), skip_rate['skip_pct'], color=COLOR_PALETTE[13])
for bar, v in zip(bars, skip_rate['skip_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_xlabel('Jahr')
ax.set_ylabel('Skip-Rate (%)')
ax.set_title('⏭️ Skip-Rate pro Jahr', fontsize=16, fontweight='bold')
ax.set_ylim(0, max(skip_rate['skip_pct']) * 1.3)
fig.savefig(out / 'skip_rate_by_year.png')
plt.show()

## Shuffle vs. Nicht-Shuffle

In [ ]:
shuffle_by_year = (music.groupby(['year', 'shuffle'])['hours_played'].sum()
                   .reset_index()
                   .pivot(index='year', columns='shuffle', values='hours_played')
                   .fillna(0))
shuffle_by_year.columns = ['Ohne Shuffle', 'Mit Shuffle']

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
shuffle_by_year.plot(kind='bar', stacked=True, ax=ax, color=[COLOR_PRIMARY, COLOR_PALETTE[10]])
ax.set_xlabel('Jahr')
ax.set_ylabel('Stunden')
ax.set_title('🔀 Shuffle vs. Nicht-Shuffle pro Jahr', fontsize=16, fontweight='bold')
ax.legend(title='')
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(out / 'shuffle_by_year.png')
plt.show()